# 08 - Statistical Analysis of Stochastic Optimization Experiments

This notebook rigorously evaluates whether the **LLaMEA Champion** significantly outperforms classical baseline algorithms (**PSO**, **DE**, **CMA-ES**).

## Context and Research Questions
We evaluate 4 stochastic optimization algorithms on various BBOB benchmark problems. Each algorithm is executed for multiple independent runs (e.g., 30 runs) using different random seeds. The primary performance metric is the **terminal residual** (the final objective value achieved at the end of the evaluation budget). For these minimization problems, **LOWER values mean BETTER performance**.

Our statistical analysis follows this conceptual workflow to address two distinct research questions:

1. **Overall Differences (`Are any of the algorithms significantly different overall?`)**
   - We use the **Kruskal-Wallis H test**.
   - This non-parametric test determines if at least one algorithm's stochastic distribution of final residuals stochastically dominates another's.
2. **Pairwise Superiority (`Is LLaMEA better than the classical baselines?`)**
   - We use the **Mann-Whitney U test** accompanied by the **Vargha-Delaney $\hat{A}_{12}$ effect size**.
   - This allows us to assess not just statistical significance (p-value), but the practical magnitude of the performance difference (effect size).

---

## The Conceptual Chain of Statistical Inference
Before diving into the tests, we must understand *why* we don't just compare mean values. Stochastic optimization runs are random variables; comparing them requires analyzing their distributions.

- **Sample Statistic**: We compute a test statistic from our empirical samples (the 30 runs). For rank-based tests, this involves pooling all residuals, ranking them from lowest (best) to highest (worst), and analyzing the rank sums.
- **Sampling Distribution**: If the Null Hypothesis ($H_0$) is true (e.g., all algorithms perform identically), the test statistic will follow a known theoretical distribution.
- **p-value**: The p-value is the probability of observing a test statistic as extreme as, or more extreme than, the one computed from our sample, *assuming $H_0$ is true*.
- **Decision**: If $p < \alpha$ (e.g., $\alpha = 0.05$), the observed data is highly unlikely under $H_0$, so we reject $H_0$ in favor of the Alternative Hypothesis ($H_1$).

### Why Non-Parametric Tests?
Parametric tests (like ANOVA or T-tests) assume the data is normally distributed and homoscedastic (equal variances). Optimization residuals are notoriously non-normal, heavily skewed, and heteroscedastic. Therefore, we use **rank-based non-parametric tests** which do not make these distributional assumptions.

In [ ]:
import re
import io
import numpy as np
import pandas as pd
import scipy.stats as stats
from pathlib import Path
from IPython.display import display

cwd = Path('.').resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
IOH_LOGS_DIR = PROJECT_ROOT / 'data' / 'ioh_logs'

# ── Benchmark Configuration ──────────────────────────────
TARGET_PROBLEMS = [1, 8, 11, 15, 21]
DIMS            = [2, 3]
NOISE_STDS      = [0.0, 0.05, 0.1]
# ─────────────────────────────────────────────────────────

## Data Loading
We load the IOHanalyzer `.dat` files for each configuration, extracting the **final objective value (terminal residual)** for each independent run.

In [ ]:
def parse_ioh_dat_file(dat_path: Path) -> list[float]:
    """Parse a single IOH .dat file and return a list of the final objective values for each run."""
    if not dat_path.exists() or dat_path.stat().st_size == 0:
        return []
        
    text = dat_path.read_text(encoding='utf-8', errors='ignore').strip()
    if not text:
        return []
        
    final_residuals = []
    blocks = [b.strip() for b in text.split('evaluations') if b.strip()]
    
    for block in blocks:
        lines = [l for l in block.splitlines() if l.strip() and not l.strip().startswith('#') and not l.strip().startswith('raw_y')]
        if not lines:
            continue
        csv_data = 'evaluations raw_y' + chr(10) + chr(10).join(lines)
        try:
            df = pd.read_csv(io.StringIO(csv_data), sep=r'\s+')
            df['raw_y'] = pd.to_numeric(df['raw_y'], errors='coerce')
            df = df.dropna().reset_index(drop=True)
            if len(df) > 0:
                # The performance metric is the minimum achieved objective value in the run
                final_residuals.append(df['raw_y'].min())
        except Exception:
            pass
            
    return final_residuals

def load_terminal_residuals(problem_dir: Path):
    """Load all final residuals grouped by algorithm from a problem directory."""
    data = {}
    if not problem_dir.exists():
        return data
        
    for algo_dir in sorted(problem_dir.iterdir()):
        if not algo_dir.is_dir():
            continue
            
        base_name = re.sub(r'-\d+$', '', algo_dir.name)
        clean_name = base_name.split('_std')[0]
        
        if any(dk in clean_name for dk in ["dummy-llm", "failing-llm", "local-model"]):
            continue
            
        if 'llamea_champion' in clean_name or 'champion' in clean_name:
            display_name = 'LLaMEA Champion'
        elif 'llamea' in clean_name or 'qwen' in clean_name or 'llama' in clean_name or 'claude' in clean_name or 'gpt' in clean_name:
            continue # Skip non-champion llamea runs for the statistical test
        else:
            display_name = clean_name.upper()
            
        residuals = []
        for dat_file in algo_dir.rglob('*.dat'):
            residuals.extend(parse_ioh_dat_file(dat_file))
                
        if residuals:
            if display_name in data:
                data[display_name].extend(residuals)
            else:
                data[display_name] = residuals
                
    return data


## Step 1: Overall Differences (Kruskal-Wallis H Test)
**Question:** Are any of the algorithms significantly different overall?

The **Kruskal-Wallis H Test** is a rank-based non-parametric alternative to the one-way ANOVA.
- **$H_0$ (Null Hypothesis):** The population medians of all algorithms are equal (no overall difference in performance distributions).
- **$H_1$ (Alternative Hypothesis):** At least one algorithm's median is different from another.

It works by pooling all runs across all algorithms, ranking them from lowest (best) to highest (worst), and then checking if the sum of ranks for any algorithm deviates significantly from what would be expected under random assignment.

In [ ]:
def kruskal_wallis_test(algo_data: dict, alpha: float = 0.05):
    """
    Performs the Kruskal-Wallis test.
    """
    if len(algo_data) < 2:
        return None, None, False
        
    samples = list(algo_data.values())
    h_stat, p_val = stats.kruskal(*samples)
    reject_null = p_val < alpha
    
    return h_stat, p_val, reject_null

## Step 2: Pairwise Superiority (Mann-Whitney U Test & Vargha-Delaney A12)
**Question:** Is LLaMEA better than the classical baselines?

If the Kruskal-Wallis test indicates a significant difference, we proceed to pairwise post-hoc tests.

### Mann-Whitney U Test
The Mann-Whitney U test is a non-parametric test used to compare differences between two independent groups.
**Important:** It is a *rank-based* test, not a test of means or medians explicitly (though often interpreted as medians if distributions have the same shape). It tests whether an observation from one distribution is stochastically more likely to be less than an observation from the other distribution.

- **$H_0$**: LLaMEA Champion and the Baseline have the same distribution.
- **$H_1$**: The distributions are different.

### Vargha-Delaney $\hat{A}_{12}$ Effect Size
Statistical significance (p-value) only tells us *if* a difference exists, not *how large* it is. With enough runs, even microscopic differences become "significant."
The **Vargha-Delaney $\hat{A}_{12}$ metric** measures the practical effect size.
- $\hat{A}_{12}$ estimates the probability that an observation from group 1 is greater than an observation from group 2.
- Since **LOWER is BETTER** in our minimization context, an $\hat{A}_{12}$ score **< 0.5** indicates that Algorithm 1 (LLaMEA) generally yields lower (better) residuals than Algorithm 2 (Baseline).
- Magnitudes: ~0.56 / 0.44 is small, ~0.64 / 0.36 is medium, ~0.71 / 0.29 is large.

In [ ]:
def vargha_delaney_A12(m, n):
    """
    Computes the Vargha and Delaney A12 effect size.
    m: array-like, data for algorithm 1 (e.g., LLaMEA)
    n: array-like, data for algorithm 2 (e.g., Baseline)
    
    Returns the probability that an observation from 'm' is strictly greater than an observation from 'n',
    plus half the probability that they are tied.
    """
    m = np.asarray(m)
    n = np.asarray(n)
    m_len = len(m)
    n_len = len(n)
    
    if m_len == 0 or n_len == 0:
        return 0.5
        
    # Combine and rank
    r = stats.rankdata(np.concatenate([m, n]))
    
    # Sum of ranks for the first group
    r1 = np.sum(r[:m_len])
    
    # A12 calculation
    a12 = (r1 / m_len - (m_len + 1) / 2) / n_len
    return a12

def pairwise_comparisons(algo_data: dict, target_algo='LLaMEA Champion', alpha=0.05):
    """
    Performs pairwise Mann-Whitney U tests and A12 effect size calculations 
    between the target algorithm and all other algorithms.
    """
    results = []
    if target_algo not in algo_data:
        return results
        
    target_runs = algo_data[target_algo]
    
    for algo, runs in algo_data.items():
        if algo == target_algo:
            continue
            
        # Mann-Whitney U test
        stat, p_val = stats.mannwhitneyu(target_runs, runs, alternative='two-sided')
        
        # A12 effect size
        a12 = vargha_delaney_A12(target_runs, runs)
        
        # Interpretation (Lower is better!)
        if a12 < 0.5:
            winner = target_algo
            diff_str = "Better"
        elif a12 > 0.5:
            winner = algo
            diff_str = "Worse"
        else:
            winner = "Tie"
            diff_str = "Equal"
            
        sig = "Yes" if p_val < alpha else "No"
        
        results.append({
            'Baseline': algo,
            'p-value': p_val,
            'Significant ($\alpha=0.05$)': sig,
            'A12 Effect Size': a12,
            'LLaMEA Performance': diff_str
        })
        
    return pd.DataFrame(results)

## Step 3: Execution and Analysis Report
Now we apply this workflow across all dimensions, noise levels, and problems.

In [ ]:
all_pairwise_results = []

for dim in DIMS:
    for noise_std in NOISE_STDS:
        print(f"\n{'='*60}")
        print(f"ANALYZING {dim}D - Noise Std {noise_std}")
        print(f"{'='*60}")
        
        for p_id in TARGET_PROBLEMS:
            prob_dir = IOH_LOGS_DIR / f"{dim}D" / f"std_{noise_std}" / f"f{p_id}"
            algo_data = load_terminal_residuals(prob_dir)
            
            if len(algo_data) < 2:
                continue
                
            print(f"\n--- Problem f{p_id} ---")
            for algo, runs in algo_data.items():
                print(f"{algo}: {len(runs)} independent runs")
            
            # 1. Overall Difference (Kruskal-Wallis)
            h_stat, kw_pval, is_sig_overall = kruskal_wallis_test(algo_data)
            print(f"\n[Kruskal-Wallis] H-statistic: {h_stat:.4f}, p-value: {kw_pval:.2e}")
            
            if not is_sig_overall:
                print("-> NO significant overall difference between algorithms.")
                continue
                
            print("-> Significant overall difference DETECTED. Proceeding to pairwise comparisons.")
            
            # 2. Pairwise Superiority (Mann-Whitney U & A12)
            if 'LLaMEA Champion' in algo_data:
                df_pairwise = pairwise_comparisons(algo_data, target_algo='LLaMEA Champion')
                if not df_pairwise.empty:
                    # Add metadata for aggregation
                    df_pairwise.insert(0, 'Noise Std', noise_std)
                    df_pairwise.insert(0, 'Dim', dim)
                    df_pairwise.insert(0, 'Problem', f'f{p_id}')
                    all_pairwise_results.append(df_pairwise)
                    
                    display(df_pairwise.drop(columns=['Problem', 'Dim', 'Noise Std']))
            else:
                print("-> LLaMEA Champion data missing for this problem.")


## Conclusion and Summary
The analysis validates the empirical performance through rank-based hypothesis testing.

- If $\hat{A}_{12} < 0.5$ and the p-value $< 0.05$, **LLaMEA Champion statistically significantly outperforms the baseline**.
- The use of Kruskal-Wallis followed by Mann-Whitney U ensures our conclusions are robust to the severe non-normality and heteroscedasticity inherent in stochastic optimization terminal residuals.

In [ ]:
if all_pairwise_results:
    final_df = pd.concat(all_pairwise_results, ignore_index=True)
    display(final_df)
    # Optional: Save to CSV
    # final_df.to_csv(PROJECT_ROOT / 'statistical_analysis_summary.csv', index=False)
else:
    print("No sufficient data for pairwise comparisons.")